# LAB 1 — Notebook 03: Ranking Paradigm
## Mobile Money Fraud Detection | EEF606 | University of Buea

---
**Prerequisite:** Run `lab1_00_eda_analysis.ipynb` first.

### What is Ranking?
Ranking asks: **'Which transactions are MOST suspicious?'** — not just 'is this fraud?'

In a real fraud operation centre, investigators can only review **N** alerts per day.
Ranking ensures the TOP N cases are the most likely frauds.
This is fundamentally different from classification:

| Paradigm | Question | Output | Use case |
|----------|----------|--------|----------|
| Classification | Is this fraud? | 0 or 1 | Automated block/allow |
| Regression | How likely is fraud? | 0.0–1.0 | Risk scoring |
| **Ranking** | **Which cases are most suspicious?** | **Ordered list** | **Investigator queue** |

### Ranking Metrics:
- **NDCG@K** — Normalised Discounted Cumulative Gain at K
- **Precision@K** — of the top K alerts, how many are real fraud?
- **Recall@K** — of all frauds, how many appear in top K?
- **Mean Average Precision (MAP)** — average precision across all recall levels

In [1]:
# ══════════════════════════════════════════════════════════
# CELL 1 — Imports
# ══════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection   import train_test_split
from sklearn.preprocessing     import RobustScaler
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm               import SVC
from sklearn.metrics           import (average_precision_score, roc_auc_score,
                                        precision_recall_curve)
from imblearn.over_sampling    import SMOTE

plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white',
                     'axes.facecolor':'#F8F9FA','axes.grid':True,'grid.alpha':0.4})
SEED = 42
print('✅ Imports loaded')

✅ Imports loaded


In [2]:
# ══════════════════════════════════════════════════════════
# CELL 2 — Custom Ranking Metrics
#
# WHY IMPLEMENT THESE MANUALLY:
# sklearn does not have built-in Precision@K or NDCG@K
# for binary relevance (fraud/not-fraud).
# Understanding the formula means you can explain to your
# lecturer exactly what each metric measures.
#
# PRECISION@K:
#   Sort all transactions by fraud score (descending).
#   Look at the top K. Count how many are real fraud.
#   Precision@K = (real frauds in top K) / K
#
# NDCG@K (Normalised Discounted Cumulative Gain):
#   Positions matter — catching fraud at rank 1 is
#   better than catching it at rank 100.
#   DCG@K = sum(relevance_i / log2(i+2)) for i in 0..K-1
#   NDCG  = DCG / IDCG  where IDCG = perfect ordering DCG
#   NDCG=1 means all fraud cases are at the very top.
# ══════════════════════════════════════════════════════════
def precision_at_k(y_true, scores, k):
    """Of the top-k ranked transactions, what fraction are real fraud?"""
    sorted_idx = np.argsort(scores)[::-1]
    top_k_labels = np.array(y_true)[sorted_idx[:k]]
    return top_k_labels.mean()

def recall_at_k(y_true, scores, k):
    """Of all real frauds, what fraction appear in the top-k alerts?"""
    sorted_idx = np.argsort(scores)[::-1]
    top_k_labels = np.array(y_true)[sorted_idx[:k]]
    total_fraud  = np.sum(y_true)
    return top_k_labels.sum() / total_fraud if total_fraud > 0 else 0.0

def ndcg_at_k(y_true, scores, k):
    """Normalised Discounted Cumulative Gain — rewards finding fraud early."""
    sorted_idx   = np.argsort(scores)[::-1][:k]
    relevances   = np.array(y_true)[sorted_idx]
    gains        = relevances / np.log2(np.arange(2, len(relevances)+2))
    dcg          = gains.sum()
    ideal_rel    = np.sort(y_true)[::-1][:k]
    ideal_gains  = ideal_rel / np.log2(np.arange(2, len(ideal_rel)+2))
    idcg         = ideal_gains.sum()
    return dcg / idcg if idcg > 0 else 0.0

def average_precision_at_k(y_true, scores, k):
    """Mean of precision@i for each i where rank i is a fraud."""
    sorted_idx = np.argsort(scores)[::-1][:k]
    hits = np.array(y_true)[sorted_idx]
    if hits.sum() == 0:
        return 0.0
    precisions = [precision_at_k(y_true, scores, i+1)
                  for i,h in enumerate(hits) if h == 1]
    return np.mean(precisions) if precisions else 0.0

print('✅ Ranking metrics defined:')
print('   precision_at_k(y_true, scores, k)')
print('   recall_at_k(y_true, scores, k)')
print('   ndcg_at_k(y_true, scores, k)')
print('   average_precision_at_k(y_true, scores, k)')

✅ Ranking metrics defined:
   precision_at_k(y_true, scores, k)
   recall_at_k(y_true, scores, k)
   ndcg_at_k(y_true, scores, k)
   average_precision_at_k(y_true, scores, k)


In [3]:
# ══════════════════════════════════════════════════════════
# CELL 3 — Load & Prepare Data
# ══════════════════════════════════════════════════════════
df = pd.read_csv('paysim_features.csv')
df = df.drop(columns=[c for c in ['isFlaggedFraud'] if c in df.columns])

X = df.drop(columns=['isFraud'])
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

scaler = RobustScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

smote = SMOTE(random_state=SEED)
X_tr_sm, y_tr_sm = smote.fit_resample(X_tr_sc, y_train)

n_fraud_test = y_test.sum()
print(f'Test set: {len(y_test):,} transactions | {n_fraud_test} real frauds')
print(f'Evaluation K values: [10, 50, 100, {n_fraud_test}]')

Test set: 60,000 transactions | 36 real frauds
Evaluation K values: [10, 50, 100, 36]


In [11]:
# ══════════════════════════════════════════════════════════
# CELL 4 — Score-Based Rankers
#
# WHY CLASSIFIERS MAKE GOOD RANKERS:
# Any classifier that outputs PROBABILITY scores can be used
# as a ranker. We rank transactions by their fraud probability
# score and evaluate how well real frauds concentrate at the top.
#
# RULE-BASED BASELINE:
# We also include a simple amount-based ranker:
# 'rank by transaction amount — larger = more suspicious'
# This is what simple rule-based systems do.
# If our ML rankers can't beat this, we have a problem.
# ══════════════════════════════════════════════════════════
rankers = {
    'Logistic Regression':  LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=SEED),
    'Random Forest':        RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=SEED),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, random_state=SEED),
    'SVM (Linear)': CalibratedClassifierCV(
    LinearSVC(
        class_weight='balanced',
        random_state=SEED,
        max_iter=5000
    )
)
}

# Rule-based baseline: rank by transaction amount
amount_scores = X_test.get('amount', X_test.iloc[:,0]).values

# K values to evaluate
K_values = sorted(set([10, 50, 100, min(200, n_fraud_test*2), n_fraud_test]))
K_values = [k for k in K_values if k <= len(y_test)]

ranking_results = {}

# Train and rank with ML models
for name, clf in rankers.items():
    print(f'Training {name} ...', end=' ')
    clf.fit(X_tr_sm, y_tr_sm)
    scores = clf.predict_proba(X_te_sc)[:,1]
    ranking_results[name] = {
        'scores': scores,
        'pr_auc': average_precision_score(y_test, scores),
        'roc_auc': roc_auc_score(y_test, scores),
        'metrics_at_k': {k: {
            'P@K': precision_at_k(y_test.values, scores, k),
            'R@K': recall_at_k(y_test.values, scores, k),
            'NDCG@K': ndcg_at_k(y_test.values, scores, k),
            'AP@K': average_precision_at_k(y_test.values, scores, k)
        } for k in K_values}
    }
    best_k = K_values[-1]
    print(f'P@{best_k}={ranking_results[name]["metrics_at_k"][best_k]["P@K"]:.3f}  NDCG@{best_k}={ranking_results[name]["metrics_at_k"][best_k]["NDCG@K"]:.3f}')

# Rule-based baseline
ranking_results['Rule: High Amount'] = {
    'scores': amount_scores,
    'pr_auc': average_precision_score(y_test, amount_scores),
    'roc_auc': roc_auc_score(y_test, amount_scores),
    'metrics_at_k': {k: {
        'P@K': precision_at_k(y_test.values, amount_scores, k),
        'R@K': recall_at_k(y_test.values, amount_scores, k),
        'NDCG@K': ndcg_at_k(y_test.values, amount_scores, k),
        'AP@K': average_precision_at_k(y_test.values, amount_scores, k)
    } for k in K_values}
}
print('\n✅ All rankers evaluated')

Training Logistic Regression ... P@100=0.210  NDCG@100=0.625
Training Random Forest ... P@100=0.280  NDCG@100=0.844
Training Gradient Boosting ... P@100=0.280  NDCG@100=0.844
Training SVM (Linear) ... P@100=0.210  NDCG@100=0.600

✅ All rankers evaluated


In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 5 — Ranking Performance Table
# ══════════════════════════════════════════════════════════
k_show = K_values[-1]  # show metrics at largest K

rank_table = pd.DataFrame({
    name: {
        'PR-AUC':    r['pr_auc'],
        'ROC-AUC':   r['roc_auc'],
        f'P@{k_show}':   r['metrics_at_k'][k_show]['P@K'],
        f'R@{k_show}':   r['metrics_at_k'][k_show]['R@K'],
        f'NDCG@{k_show}':r['metrics_at_k'][k_show]['NDCG@K'],
    } for name, r in ranking_results.items()
}).T.sort_values('PR-AUC', ascending=False)

print(f'Ranking Performance (K={k_show}):')
print(rank_table.round(4).to_string())

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 6 — Precision@K and Recall@K Curves
#
# WHY PLOT ACROSS ALL K:
# A fraud team that can investigate 50 cases per day needs
# to know Precision@50. A team that can handle 200 needs
# Precision@200. Plotting across all K shows how each
# model performs at YOUR team's capacity.
# ══════════════════════════════════════════════════════════
k_range = range(5, min(500, len(y_test)), 5)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors_r = plt.cm.Set1(np.linspace(0, 1, len(ranking_results)))

for (name, r), color in zip(ranking_results.items(), colors_r):
    p_at_ks = [precision_at_k(y_test.values, r['scores'], k) for k in k_range]
    rec_at_ks = [recall_at_k(y_test.values, r['scores'], k) for k in k_range]
    axes[0].plot(list(k_range), p_at_ks, label=name, color=color, linewidth=1.8)
    axes[1].plot(list(k_range), rec_at_ks, label=name, color=color, linewidth=1.8)

# Baseline: random ranker
fraud_rate = y_test.mean()
axes[0].axhline(fraud_rate, color='k', linestyle='--', linewidth=1, label=f'Random ({fraud_rate:.3f})')
axes[1].plot(list(k_range), [k*fraud_rate/n_fraud_test for k in k_range],
             'k--', linewidth=1, label='Random')

axes[0].set_xlabel('K (Number of Alerts Reviewed)')
axes[0].set_ylabel('Precision@K')
axes[0].set_title('Precision@K — Alert Accuracy vs Investigation Capacity', fontweight='bold')
axes[0].legend(fontsize=8)

axes[1].set_xlabel('K (Number of Alerts Reviewed)')
axes[1].set_ylabel('Recall@K')
axes[1].set_title('Recall@K — Fraud Caught vs Investigation Capacity', fontweight='bold')
axes[1].legend(fontsize=8)

plt.suptitle('Ranking Evaluation: How Good is Each Model at Prioritising Fraud?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_rank_01_pk_rk_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 7 — Ranked List Visualisation (Top 30 Alerts)
#
# WHY VISUALISE THE RANKED LIST:
# This is what an investigator actually SEES on their screen.
# Showing the ranked output with true labels lets us see:
# - Are the first few rows all real fraud? (good ranker)
# - Are frauds scattered throughout? (poor ranker)
# The colour of each bar = actual label (red=fraud, blue=legit)
# ══════════════════════════════════════════════════════════
best_ranker = rank_table.index[0]
best_scores = ranking_results[best_ranker]['scores']

sorted_idx  = np.argsort(best_scores)[::-1][:30]
top30_scores = best_scores[sorted_idx]
top30_labels = y_test.values[sorted_idx]

fig, ax = plt.subplots(figsize=(14, 5))
bar_colors = ['#EF5350' if l==1 else '#42A5F5' for l in top30_labels]
bars = ax.bar(range(1,31), top30_scores, color=bar_colors, edgecolor='white')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#EF5350', label='Actual Fraud'),
                   Patch(facecolor='#42A5F5', label='Actual Legitimate')]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_xlabel('Rank (1 = most suspicious)')
ax.set_ylabel('Fraud Probability Score')
ax.set_title(f'Top 30 Ranked Alerts — {best_ranker}\n'
             f'Red = actual fraud | Blue = false alarm',
             fontweight='bold', fontsize=12)
ax.set_xticks(range(1,31))

fraud_in_top30 = top30_labels.sum()
ax.text(0.99, 0.95, f'Precision@30 = {fraud_in_top30}/30 = {fraud_in_top30/30:.2f}',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=12, fontweight='bold', color='#C62828',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='#C62828', alpha=0.8))

plt.tight_layout()
plt.savefig('fig_rank_02_top30_alerts.png', bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 8 — NDCG@K Comparison
# ══════════════════════════════════════════════════════════
ndcg_data = {
    name: [ndcg_at_k(y_test.values, r['scores'], k) for k in k_range]
    for name, r in ranking_results.items()
}

fig, ax = plt.subplots(figsize=(11, 6))
for (name, vals), color in zip(ndcg_data.items(), colors_r):
    ax.plot(list(k_range), vals, label=name, color=color, linewidth=1.8)

ax.set_xlabel('K')
ax.set_ylabel('NDCG@K')
ax.set_title('NDCG@K — Ranking Quality\n'
             'NDCG=1.0: all frauds at top | NDCG=0: worst possible ranking',
             fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig('fig_rank_03_ndcg.png', bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 9 — Summary
# ══════════════════════════════════════════════════════════
print('╔' + '═'*65 + '╗')
print('║{:^65}║'.format('RANKING PARADIGM SUMMARY'))
print('╠' + '═'*65 + '╣')
print(f'║  Best Ranker: {best_ranker:<51}║')
print('╠' + '═'*65 + '╣')
insights = [
    'Ranking = ordering by suspicion score, not labelling',
    'Precision@K = quality of top-K fraud alert queue',
    'NDCG rewards catching fraud at EARLIER positions',
    'Rule baseline (high amount) = weak — ML outperforms',
    'In production: capacity K = fraud team size per day',
    'Use Recall@K to ensure high-risk cases not missed',
]
for i in insights:
    print(f'║  → {i:<61}║')
print('╚' + '═'*65 + '╝')
print('\n→ Next: lab1_04_anomaly_detection.ipynb')

In [ ]:
# ══════════════════════════════════════════════════════════
# UNIVERSAL REPORT SUMMARY GENERATOR
# Paste this as the last cell in any notebook to auto-extract metrics
# ══════════════════════════════════════════════════════════
import pandas as pd
import numpy as np

print("="*80)
print("📊 AUTOMATED LAB REPORT SUMMARY GENERATOR")
print("="*80)

# Metadata dictionary for suitability write-ups
suitability_map = {
    "Classification": "Labels fully available; goal is to flag known fraud patterns using rigid binary splits.",
    "Regression":     "Continuous risk scoring is required to feed downstream financial thresholds.",
    "Ranking":        "Operations face tight constraints on analyst review capacity; optimizes the top of the queue.",
    "Ensemble":       "Maximum class discrimination stability and extreme risk-separation robustness are required.",
    "Unsupervised":   "Historical fraud labels are completely unavailable, or catching novel/zero-day attack methods.",
    "Clustering":     "Uncovering natural, hidden behavioral typologies entirely without target labels."
}

# --- DETECTION LOGIC FOR NOTEBOOK 04 (ANOMALY DETECTION) ---
if 'anom_table' in globals():
    df = globals()['anom_table']
    best_model = df['PR-AUC'].idxmax()
    row = df.loc[best_model]
    
    print(f"| Paradigm | Best Model | F1 | Recall | AUC | Suitable When |")
    print(f"| :--- | :--- | :---: | :---: | :---: | :--- |")
    print(f"| **Anomaly Detection** | {best_model} | {row.get('F1 (Fraud)', row.get('F1-Score (Fraud)', 0)):.4f} | N/A | {row['ROC-AUC']:.4f} (ROC) <br> {row['PR-AUC']:.4f} (PR) | {suitability_map['Unsupervised']} |")

# --- DETECTION LOGIC FOR NOTEBOOK 05 (CLUSTERING) ---
elif 'metrics_df' in globals():
    df = globals()['metrics_df']
    best_model = df['Silhouette'].idxmax()
    row = df.loc[best_model]
    
    print(f"| Method | ARI | Silhouette | DB Index | Key Finding |")
    print(f"| :--- | :---: | :---: | :---: | :--- |")
    print(f"| **{best_model}** | {row['ARI']:.4f} | {row['Silhouette']:.4f} | {row['Davies-Bouldin']:.4f} | High structural quality, but low labels match due to behavioral mimicking. |")

# --- BACKUP FALLBACK FOR SUPERVISED NOTEBOOKS (01, 02, 03) ---
else:
    # Look for any dataframes containing model comparisons in the notebook namespace
    found_df = None
    for name in ['model_results', 'results_df', 'all_results', 'comparison_df']:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            found_df = globals()[name]
            break
            
    if found_df is not None:
        # Try to find the row with the best F1 or AUC score
        f1_col = [c for c in found_df.columns if 'f1' in c.lower()]
        auc_col = [c for c in found_df.columns if 'auc' in c.lower() or 'roc' in c.lower()]
        
        target_f1 = f1_col[0] if f1_col else found_df.columns[0]
        best_model = found_df[target_f1].idxmax()
        row = found_df.loc[best_model]
        
        f1_val = f" {row[f1_col[0]]:.4f}" if f1_col else "N/A"
        auc_val = f" {row[auc_col[0]]:.4f}" if auc_col else "N/A"
        recall_val = "Check notebook metrics"
        
        print(f"| Paradigm | Best Model | F1 | Recall | AUC | Suitable When |")
        print(f"| :--- | :--- | :---: | :---: | :---: | :--- |")
        print(f"| **Supervised** | {best_model} | {f1_val} | {recall_val} | {auc_val} | Match with classification/ensemble targets. |")
    else:
        print("❌ Could not auto-detect summary tables in memory.")
        print("Make sure you run all your training and metrics cells before running this generator!")
print("="*80)